In [6]:

import pandas as pd
import os

# --------------------------------------------------
# 1. LECTURA DE ARCHIVOS LIMPIOS
# --------------------------------------------------

df_modis = pd.read_csv("MODIS_limpio.csv")
df_viirs = pd.read_csv("VIIRS_limpio.csv")

# --------------------------------------------------
# 2. IDENTIFICACIÓN DEL SENSOR
# --------------------------------------------------

df_modis["sensor"] = "MODIS"
df_viirs["sensor"] = "VIIRS"

# Unificar ambos archivos

detecciones = pd.concat(
    [df_modis, df_viirs],
    ignore_index=True
)


# --------------------------------------------------
# 3. NORMALIZACIÓN DE VARIABLES
# --------------------------------------------------

detecciones = detecciones.rename(columns={
    "acq_date": "fecha",
    "acq_time": "hora"
})

# Convertir variables numéricas

for columna in ["latitude", "longitude", "frp_mw"]:
    detecciones[columna] = pd.to_numeric(
        detecciones[columna],
        errors="coerce"
    )

# Normalizar fecha

detecciones["fecha"] = pd.to_datetime(
    detecciones["fecha"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

# Normalizar hora

detecciones["hora"] = (
    detecciones["hora"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(4)
)

# Normalizar confianza

detecciones["confidence"] = (
    detecciones["confidence"]
    .astype("string")
    .str.strip()
    .str.lower()
)


# --------------------------------------------------
# 4. SELECCIÓN DE VARIABLES
# --------------------------------------------------

columnas_finales = [
    "latitude",
    "longitude",
    "fecha",
    "hora",
    "frp_mw",
    "confidence",
    "sensor",
    "satellite",
    "instrument"
]

detecciones = detecciones[columnas_finales]

# Eliminar registros inválidos

detecciones = detecciones.dropna(
    subset=[
        "latitude",
        "longitude",
        "fecha",
        "frp_mw"
    ]
)

# Validar coordenadas y FRP

detecciones = detecciones[
    detecciones["latitude"].between(-90, 90)
    & detecciones["longitude"].between(-180, 180)
    & (detecciones["frp_mw"] >= 0)
].copy()

# Eliminar duplicados

detecciones = (
    detecciones
    .drop_duplicates()
    .reset_index(drop=True)
)


# --------------------------------------------------
# 5. EXPORTACIÓN
# --------------------------------------------------

os.makedirs(
    "/content/datos_armonizados",
    exist_ok=True
)

detecciones.to_csv(
    "/content/datos_armonizados/detecciones.csv",
    index=False,
    encoding="utf-8-sig"
)


# --------------------------------------------------
# 6. RESUMEN
# --------------------------------------------------

print("DETECCIONES ARMONIZADAS")

print("Total de detecciones:", len(detecciones))

print("\nDetecciones por sensor:")
print(detecciones["sensor"].value_counts())

print("\nDetecciones por satélite:")
print(detecciones["satellite"].value_counts())

print("\nFRP promedio:", detecciones["frp_mw"].mean())
print("FRP máximo:", detecciones["frp_mw"].max())

display(detecciones.head())


DETECCIONES ARMONIZADAS
Total de detecciones: 694852

Detecciones por sensor:
sensor
VIIRS    556146
MODIS    138706
Name: count, dtype: int64

Detecciones por satélite:
satellite
SNPP     323523
N20      173428
AQUA      97324
N21       59195
TERRA     41382
Name: count, dtype: int64

FRP promedio: 26.369632151882698
FRP máximo: 12844.8


,latitude,longitude,fecha,hora,frp_mw,confidence,sensor,satellite,instrument
0,-37.0977,-71.9882,2000-11-01,1511,6.9,nominal,MODIS,TERRA,MODIS
1,-37.0993,-71.9766,2000-11-01,1511,9.1,nominal,MODIS,TERRA,MODIS
2,-53.0085,-69.2269,2000-11-07,1439,5.4,nominal,MODIS,TERRA,MODIS
3,-35.8858,-72.6163,2000-11-08,1517,7.2,nominal,MODIS,TERRA,MODIS
4,-28.2195,-69.6724,2000-11-12,1450,15.8,nominal,MODIS,TERRA,MODIS


In [9]:

import pandas as pd
import geopandas as gpd
import numpy as np
import os

from shapely.geometry import box


# --------------------------------------------------
# 1. LECTURA DE DETECCIONES
# --------------------------------------------------

df = pd.read_csv(
    "datos_armonizados/detecciones.csv"
)

# Validar columnas necesarias

columnas_requeridas = [
    "latitude",
    "longitude",
    "fecha",
    "frp_mw"
]

for columna in columnas_requeridas:
    if columna not in df.columns:
        raise ValueError(
            f"Falta la columna requerida: {columna}"
        )

# Convertir variables numéricas

df["latitude"] = pd.to_numeric(
    df["latitude"], errors="coerce"
)

df["longitude"] = pd.to_numeric(
    df["longitude"], errors="coerce"
)

df["frp_mw"] = pd.to_numeric(
    df["frp_mw"], errors="coerce"
)

df["fecha"] = pd.to_datetime(
    df["fecha"], errors="coerce"
)

# Eliminar registros inválidos

df = df.dropna(
    subset=["latitude", "longitude", "fecha", "frp_mw"]
)

df = df[
    df["latitude"].between(-90, 90)
    & df["longitude"].between(-180, 180)
    & (df["frp_mw"] >= 0)
].copy()

if df.empty:
    raise ValueError(
        "No existen detecciones válidas para construir los polígonos."
    )


# --------------------------------------------------
# 2. CONFIGURACIÓN
# --------------------------------------------------

TAMANO_CELDA_M = 5000
FRECUENCIA = "D"


# --------------------------------------------------
# 3. CONVERSIÓN DE COORDENADAS
# --------------------------------------------------

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(
        df["longitude"],
        df["latitude"]
    ),
    crs="EPSG:4326"
)

gdf = gdf.to_crs("EPSG:6933")

gdf["coord_x"] = gdf.geometry.x
gdf["coord_y"] = gdf.geometry.y


# --------------------------------------------------
# 4. ASIGNACIÓN A CELDAS DE 5 X 5 KM
# --------------------------------------------------

gdf["celda_x"] = (
    np.floor(gdf["coord_x"] / TAMANO_CELDA_M)
    * TAMANO_CELDA_M
)

gdf["celda_y"] = (
    np.floor(gdf["coord_y"] / TAMANO_CELDA_M)
    * TAMANO_CELDA_M
)


# --------------------------------------------------
# 5. DEFINICIÓN DEL PERÍODO
# --------------------------------------------------

gdf["periodo"] = (
    gdf["fecha"]
    .dt.to_period(FRECUENCIA)
    .astype(str)
)

columnas_agrupacion = [
    "celda_x",
    "celda_y",
    "periodo"
]


# --------------------------------------------------
# 6. AGRUPACIÓN Y MÉTRICAS DE FRP
# --------------------------------------------------

poligonos = (
    gdf.groupby(columnas_agrupacion)
    .agg(
        cantidad_detecciones=("frp_mw", "size"),
        frp_promedio_mw=("frp_mw", "mean"),
        frp_maximo_mw=("frp_mw", "max"),
        frp_total_mw=("frp_mw", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 7. CLASIFICACIÓN DEL NIVEL DE ACTIVIDAD
# --------------------------------------------------

percentil = poligonos["frp_total_mw"].rank(
    method="average",
    pct=True
)

poligonos["nivel_actividad"] = np.select(
    [
        percentil <= 0.33,
        percentil <= 0.66
    ],
    ["bajo", "medio"],
    default="alto"
)


# --------------------------------------------------
# 8. CONSTRUCCIÓN DE GEOMETRÍAS
# --------------------------------------------------

geometrias = []

for _, fila in poligonos.iterrows():

    geometria = box(
        fila["celda_x"],
        fila["celda_y"],
        fila["celda_x"] + TAMANO_CELDA_M,
        fila["celda_y"] + TAMANO_CELDA_M
    )

    geometrias.append(geometria)

poligonos = gpd.GeoDataFrame(
    poligonos,
    geometry=geometrias,
    crs="EPSG:6933"
)

poligonos = poligonos.to_crs("EPSG:4326")


# --------------------------------------------------
# 9. IDENTIFICADORES
# --------------------------------------------------

poligonos = poligonos.sort_values(
    columnas_agrupacion
).reset_index(drop=True)

poligonos.insert(
    0,
    "id_poligono",
    [
        f"POL_{i:08d}"
        for i in range(1, len(poligonos) + 1)
    ]
)


# --------------------------------------------------
# 10. EXPORTACIÓN
# --------------------------------------------------

os.makedirs("datos_armonizados", exist_ok=True)

# Exportar GeoJSON

poligonos.to_file(
    "datos_armonizados/poligonos.geojson",
    driver="GeoJSON"
)

# Exportar CSV

poligonos_csv = poligonos.copy()

poligonos_csv["geometry"] = (
    poligonos_csv.geometry.apply(
        lambda geometria: geometria.wkt
    )
)

poligonos_csv.drop(
    columns=["celda_x", "celda_y"],
    inplace=True
)

poligonos_csv.to_csv(
    "datos_armonizados/poligonos.csv",
    index=False,
    encoding="utf-8-sig"
)


# --------------------------------------------------
# 11. RESUMEN
# --------------------------------------------------

print("GENERACIÓN DE POLÍGONOS COMPLETADA")

print("Total de polígonos:", len(poligonos))

print("\nNiveles de actividad:")
print(poligonos["nivel_actividad"].value_counts())

print("\nTotal de detecciones agrupadas:")
print(poligonos["cantidad_detecciones"].sum())

print("\nFRP total acumulado:")
print(poligonos["frp_total_mw"].sum())

print("\nFRP máximo registrado:")
print(poligonos["frp_maximo_mw"].max())

print("\nColumnas finales:")
print(poligonos_csv.columns.tolist())

display(
    poligonos.drop(
        columns=["geometry", "celda_x", "celda_y"]
    ).head()
)


/tmp/ipykernel_1400/1109832228.py:223: UserWarning: Geometry column does not contain geometry.
  poligonos_csv["geometry"] = (


GENERACIÓN DE POLÍGONOS COMPLETADA
Total de polígonos: 197834

Niveles de actividad:
nivel_actividad
alto     67278
medio    65289
bajo     65267
Name: count, dtype: int64

Total de detecciones agrupadas:
694852

FRP total acumulado:
18322991.64

FRP máximo registrado:
12844.8

Columnas finales:
['id_poligono', 'periodo', 'cantidad_detecciones', 'frp_promedio_mw', 'frp_maximo_mw', 'frp_total_mw', 'nivel_actividad', 'geometry']


,id_poligono,periodo,cantidad_detecciones,frp_promedio_mw,frp_maximo_mw,frp_total_mw,nivel_actividad
0,POL_00000001,2003-10-22,1,11.000,11.0,11.00,medio
1,POL_00000002,2014-01-06,1,23.100,23.1,23.10,alto
2,POL_00000003,2016-02-20,6,19.645,36.7,117.87,alto
3,POL_00000004,2002-10-09,1,9.100,9.1,9.10,medio
4,POL_00000005,2002-10-13,1,10.700,10.7,10.70,medio
